# 0. Prepare SCOPe40 工作区

把官方 SCOPe 2.08 的 40% identity PDB-style 子集整理到项目 `work/`：

| 产出 | 路径 |
|------|------|
| GT FASTA | `work/GT_fasta/DB_aa.fasta`、`DB_di.fasta` |
| 对比库 | `work/DB/foldseek_DB`、`work/DB/mmseqs_DB` |
| 标签 | `work/lable/scop_lookup.tsv` |
| 二进制 | `bin/foldseek`、`bin/mmseqs` |

中间文件在 `tmp/`。用户随后根据 `GT_fasta` 跑模型，把结果放入 `work/aa2di_fasta/`（及可选 `work/di2aa_fasta/`）。

全部逻辑在本 notebook；重来：`rm -rf tmp work/DB work/GT_fasta work/lable bin`（保留 `aa2di_fasta` / `di2aa_fasta` 若需要）。

下一步：放入预测 FASTA → `1.init.ipynb`


## 1. 配置

路径、URL、方法表见 [`config.py`](config.py)。本 cell 导入共享配置，并定义 prepare 专用路径 / 跳过策略 / 线程数。


In [1]:
from __future__ import annotations

import csv
import hashlib
import json
import os
import shutil
import subprocess
import tarfile
from pathlib import Path

from config import (
    AA_FASTA,
    BIN_DIR,
    DBS_DIR,
    FOLDSEEK_BIN,
    FOLDSEEK_GT_DIR,
    FOLDSEEK_URL,
    GT_DI_FASTA,
    GT_FASTA_DIR,
    HF_BASE,
    LABEL_DIR,
    MMSEQS_BIN,
    MMSEQS_GT_DIR,
    MMSEQS_URL,
    PREPARE_THREADS,
    PROJECT_ROOT,
    SCOP_LOOKUP,
    TEMP,
    WORK_BUNDLE,
    WORK_DIR,
    cleanup_tmp,
    ensure_work_dirs,
    require_project_root,
)

ROOT = require_project_root("0.prepare.ipynb")
assert ROOT == PROJECT_ROOT

# Intermediate build under tmp/ (not the final work/ layout)
TMP_WORK = TEMP / "work"
SOURCE_ARCHIVE = TEMP / "pdbstyle-sel-gs-bib-40-2.08.tgz"
SCOP_CLA = TEMP / "dir.cla.scope.2.08-stable.txt"
SCOP_DES = TEMP / "dir.des.scope.2.08-stable.txt"
EXTRACTED = TEMP / "pdbstyle-2.08"

SINGLE = TMP_WORK / "SCOPe40"
MULTI = TMP_WORK / "SCOPe40-multi"
OTHER = TMP_WORK / "SCOPe40-other"
BUILD_DB = TMP_WORK / "FoldseekDB"
BUILD_MMSEQS_DB = TMP_WORK / "MMseqsDB"
CLASSIFICATION = TMP_WORK / "structure_classification.tsv"  # 中间产物，不写入 work/

SKIP_EXISTING = True
THREADS = PREPARE_THREADS

ensure_work_dirs()
for path in (TEMP, TMP_WORK):
    path.mkdir(parents=True, exist_ok=True)

print("ROOT:", ROOT)
print("TEMP:", TEMP)
print("work/:", WORK_DIR)
print("GT_fasta:", GT_FASTA_DIR)
print("DB:", DBS_DIR)
print("lable:", LABEL_DIR)
print("foldseek:", FOLDSEEK_BIN)
print("mmseqs:", MMSEQS_BIN)
print(f"SKIP_EXISTING={SKIP_EXISTING}  THREADS={THREADS}")


ROOT: /hpcfs/fhome/caihuize/scope40_easy
TEMP: /hpcfs/fhome/caihuize/scope40_easy/tmp
work/: /hpcfs/fhome/caihuize/scope40_easy/work
GT_fasta: /hpcfs/fhome/caihuize/scope40_easy/work/GT_fasta
DB: /hpcfs/fhome/caihuize/scope40_easy/work/DB
lable: /hpcfs/fhome/caihuize/scope40_easy/work/lable
foldseek: /hpcfs/fhome/caihuize/scope40_easy/bin/foldseek
mmseqs: /hpcfs/fhome/caihuize/scope40_easy/bin/mmseqs
SKIP_EXISTING=True  THREADS=16


## 1. 工具函数

分类、建库、打包共用的小函数。折叠本 cell 即可；业务步骤在后面各节。


In [2]:
def run(cmd: list[str]) -> None:
    print("[CMD]", " ".join(cmd), flush=True)
    subprocess.run(cmd, check=True)


def count_models(path: Path) -> int:
    with path.open(encoding="utf-8", errors="ignore") as handle:
        return sum(line.startswith("MODEL") for line in handle)


def has_atoms(path: Path) -> bool:
    with path.open(encoding="utf-8", errors="ignore") as handle:
        for line in handle:
            if line.startswith(("ATOM", "HETATM")):
                return True
    return False


def load_scop_classes() -> dict[str, str]:
    if not SCOP_CLA.is_file():
        raise FileNotFoundError(f"缺少 SCOP 分类文件: {SCOP_CLA}")
    id_to_class: dict[str, str] = {}
    with SCOP_CLA.open(encoding="utf-8") as handle:
        for line in handle:
            if line.startswith("#") or not line.strip():
                continue
            parts = line.rstrip().split("\t")
            if len(parts) >= 4:
                id_to_class[parts[0]] = parts[3]
    return id_to_class


def resolve_scop_class(seq_id: str, id_to_class: dict[str, str]) -> str:
    """Map Foldseek FASTA id → SCOPe class; fall back parent id if chain-split."""
    if seq_id in id_to_class:
        return id_to_class[seq_id]
    if "_" in seq_id:
        parent = seq_id.rsplit("_", 1)[0]
        if parent in id_to_class:
            return id_to_class[parent]
    return ""


def extract_source(skip_existing: bool = True) -> Path:
    """Ensure pdbstyle archive is extracted under tmp/pdbstyle-2.08."""
    if skip_existing and EXTRACTED.is_dir() and any(EXTRACTED.rglob("*.ent")):
        print(f"⏭️  已解压: {EXTRACTED}")
        return EXTRACTED
    if not SOURCE_ARCHIVE.is_file():
        raise FileNotFoundError(f"缺少 SCOPe40 源压缩包: {SOURCE_ARCHIVE}")
    with tarfile.open(SOURCE_ARCHIVE, "r:gz") as tf:
        root = EXTRACTED.parent.resolve()
        for member in tf.getmembers():
            destination = (EXTRACTED.parent / member.name).resolve()
            if root not in destination.parents and destination != root:
                raise RuntimeError(f"不安全的归档路径: {member.name}")
        tf.extractall(path=EXTRACTED.parent)
    if not EXTRACTED.is_dir():
        raise FileNotFoundError(f"解压后未找到: {EXTRACTED}")
    return EXTRACTED


def classify_structures(skip_existing: bool = True) -> Path:
    """Classify .ent into single / multi / other (hardlink when possible)."""
    if (
        skip_existing
        and CLASSIFICATION.is_file()
        and SINGLE.is_dir()
        and MULTI.is_dir()
        and OTHER.is_dir()
    ):
        print(f"⏭️  分类已存在: {CLASSIFICATION}")
        return CLASSIFICATION

    id_to_class = load_scop_classes()
    for directory in (SINGLE, MULTI, OTHER):
        if directory.exists():
            shutil.rmtree(directory)
        directory.mkdir(parents=True)

    pdb_files = sorted(EXTRACTED.rglob("*.ent"))
    if not pdb_files:
        raise FileNotFoundError(f"没有找到 .ent: {EXTRACTED}")

    rows: list[tuple[str, int, str, int, str]] = []
    for index, source in enumerate(pdb_files, start=1):
        models = count_models(source)
        domain_id = source.stem
        scop_class = id_to_class.get(domain_id, "")
        in_cla = int(bool(scop_class))
        if not has_atoms(source) or not in_cla:
            group, destination = "other", OTHER
        else:
            group = "multi" if models > 1 else "single"
            destination = MULTI if group == "multi" else SINGLE
        target = destination / source.name
        try:
            os.link(source, target)
        except OSError:
            shutil.copy2(source, target)
        rows.append((source.name, models, group, in_cla, scop_class))
        if index % 1000 == 0:
            print(f"分类进度: {index:,}/{len(pdb_files):,}", flush=True)

    TMP_WORK.mkdir(parents=True, exist_ok=True)
    with CLASSIFICATION.open("w", newline="", encoding="utf-8") as handle:
        writer = csv.writer(handle, delimiter="\t")
        writer.writerow(["file", "model_count", "group", "in_cla", "scop_class"])
        writer.writerows(rows)

    print(
        f"✅ 分类: single={sum(r[2] == 'single' for r in rows):,}, "
        f"multi={sum(r[2] == 'multi' for r in rows):,}, "
        f"other={sum(r[2] == 'other' for r in rows):,}"
    )
    return CLASSIFICATION


def build_foldseek_db(skip_existing: bool = True, threads: int = 16) -> Path:
    marker = BUILD_DB / "DB"
    if skip_existing and marker.is_file():
        print(f"⏭️  Foldseek DB 已存在: {marker}")
        return marker
    if not FOLDSEEK_BIN.is_file():
        raise FileNotFoundError(f"Foldseek 不存在: {FOLDSEEK_BIN}")
    if BUILD_DB.exists():
        shutil.rmtree(BUILD_DB)
    BUILD_DB.mkdir(parents=True)

    db = str(marker)
    run([str(FOLDSEEK_BIN), "createdb", str(SINGLE), db, "--threads", str(threads)])
    run([str(FOLDSEEK_BIN), "lndb", f"{db}_h", f"{db}_ss_h"])
    run([str(FOLDSEEK_BIN), "convert2fasta", db, f"{db}_aa.fasta"])
    run([str(FOLDSEEK_BIN), "convert2fasta", f"{db}_ss", f"{db}_di.fasta"])
    return marker


def build_mmseqs_db(skip_existing: bool = True, threads: int = 16) -> Path:
    marker = BUILD_MMSEQS_DB / "DB"
    aa_fasta = BUILD_DB / "DB_aa.fasta"
    if skip_existing and marker.is_file():
        print(f"⏭️  MMseqs DB 已存在: {marker}")
        return marker
    if not MMSEQS_BIN.is_file():
        raise FileNotFoundError(f"MMseqs 不存在: {MMSEQS_BIN}")
    if not aa_fasta.is_file():
        raise FileNotFoundError(f"缺少 Foldseek AA FASTA: {aa_fasta}")
    if BUILD_MMSEQS_DB.exists():
        shutil.rmtree(BUILD_MMSEQS_DB)
    BUILD_MMSEQS_DB.mkdir(parents=True)

    run(
        [
            str(MMSEQS_BIN),
            "createdb",
            str(aa_fasta),
            str(marker),
            "--threads",
            str(threads),
        ]
    )
    return marker


def build_scop_lookup(aa_fasta: Path, output: Path | None = None) -> Path:
    output = output or SCOP_LOOKUP
    id_to_class = load_scop_classes()
    ids = [
        line[1:].split()[0]
        for line in aa_fasta.open(encoding="utf-8")
        if line.startswith(">")
    ]
    matched = 0
    output.parent.mkdir(parents=True, exist_ok=True)
    with output.open("w", encoding="utf-8") as handle:
        for seq_id in ids:
            scop_class = resolve_scop_class(seq_id, id_to_class)
            if scop_class:
                handle.write(f"{seq_id}\t{scop_class}\n")
                matched += 1
    print(f"✅ scop lookup: {output}  匹配={matched:,} 未匹配={len(ids) - matched:,}")
    return output


def _sha256(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


def _copy_db_tree(src: Path, dst: Path) -> None:
    if dst.exists():
        if dst.is_symlink() or dst.is_file():
            dst.unlink()
        else:
            shutil.rmtree(dst)
    dst.mkdir(parents=True)
    for source in src.iterdir():
        if source.name in {"DB_aa.fasta", "DB_di.fasta"}:
            continue
        target = dst / source.name
        if source.is_symlink():
            shutil.copy2(source.resolve(), target)
        elif source.is_file():
            shutil.copy2(source, target)


def install_to_work(skip_existing: bool = True) -> Path:
    """Install Foldseek/MMseqs DBs, GT FASTA, and scop_lookup into work/."""
    if skip_existing and (FOLDSEEK_GT_DIR / "DB").is_file() and AA_FASTA.is_file() and SCOP_LOOKUP.is_file():
        print(f"⏭️  work/ 已就绪: {WORK_DIR}")
        return WORK_DIR

    if not (BUILD_DB / "DB").is_file():
        raise FileNotFoundError("请先构建 Foldseek DB")
    if not (BUILD_MMSEQS_DB / "DB").is_file():
        raise FileNotFoundError("请先构建 MMseqs DB")

    ensure_work_dirs()
    _copy_db_tree(BUILD_DB, FOLDSEEK_GT_DIR)
    _copy_db_tree(BUILD_MMSEQS_DB, MMSEQS_GT_DIR)

    for name in ("DB_aa.fasta", "DB_di.fasta"):
        src = BUILD_DB / name
        if not src.is_file():
            raise FileNotFoundError(f"缺少 {src}")
        shutil.copy2(src, GT_FASTA_DIR / name)

    # 标签：仅保留 scop_lookup.tsv
    LABEL_DIR.mkdir(parents=True, exist_ok=True)
    for stale in LABEL_DIR.iterdir():
        if stale.is_file() and stale.name != SCOP_LOOKUP.name:
            stale.unlink()
    lookup = build_scop_lookup(AA_FASTA, SCOP_LOOKUP)

    print(f"✅ 已安装到 {WORK_DIR}")
    print(f"   GT_fasta: {AA_FASTA.name}, {GT_DI_FASTA.name}")
    print(f"   DB: {FOLDSEEK_GT_DIR.name}, {MMSEQS_GT_DIR.name}")
    print(f"   lable: {lookup}")
    return WORK_DIR


def make_work_bundle(skip_existing: bool = True) -> Path:
    """Optional tarball of GT_fasta + DB + lable for sharing."""
    if skip_existing and WORK_BUNDLE.is_file() and WORK_BUNDLE.stat().st_size > 0:
        print(f"⏭️  bundle 已存在: {WORK_BUNDLE}")
        return WORK_BUNDLE
    if not (FOLDSEEK_GT_DIR / "DB").is_file() or not SCOP_LOOKUP.is_file():
        raise FileNotFoundError("请先 install_to_work")
    with tarfile.open(WORK_BUNDLE, "w:gz") as tf:
        for rel in ("GT_fasta", "DB/foldseek_DB", "DB/mmseqs_DB", "lable"):
            path = WORK_DIR / rel
            tf.add(path, arcname=f"scope40_work/{rel}")
    print(f"✅ bundle: {WORK_BUNDLE} ({WORK_BUNDLE.stat().st_size / 2**20:.1f} MiB)")
    return WORK_BUNDLE


def verify_work() -> None:
    """Check required work/ artifacts exist."""
    checks = {
        "foldseek DB": FOLDSEEK_GT_DIR / "DB",
        "mmseqs DB": MMSEQS_GT_DIR / "DB",
        "DB_aa.fasta": AA_FASTA,
        "DB_di.fasta": GT_DI_FASTA,
        "scop_lookup": SCOP_LOOKUP,
    }
    missing = []
    for name, path in checks.items():
        ok = path.is_file()
        print(("✅" if ok else "❌"), f"{name}: {path}")
        if not ok:
            missing.append(name)
    if missing:
        raise RuntimeError(f"校验失败，缺失: {missing}")
    n = sum(1 for _ in SCOP_LOOKUP.open())
    print(f"verify OK — scop_lookup 行数={n:,}")


print("helpers ready:", ", ".join([
    "extract_source", "classify_structures", "build_foldseek_db",
    "build_mmseqs_db", "install_to_work", "make_work_bundle", "verify_work",
]))


helpers ready: extract_source, classify_structures, build_foldseek_db, build_mmseqs_db, install_to_work, make_work_bundle, verify_work


## 2. 下载 Foldseek / MMseqs（Linux AVX2）

**只需运行一次**（`wget -nc` 可跳过重复下载）。

1. 压缩包下载并解压到 `tmp/`
2. 将可执行文件复制到项目根 `bin/`（供全项目使用）


In [3]:
# Foldseek
!cd tmp && wget -nc {FOLDSEEK_URL}
!cd tmp && tar xvzf foldseek-linux-avx2.tar.gz
!mkdir -p bin
!cp -f tmp/foldseek/bin/foldseek bin/foldseek
!chmod +x bin/foldseek
!./bin/foldseek version

# MMseqs2
!cd tmp && wget -nc {MMSEQS_URL}
!cd tmp && tar xvzf mmseqs-linux-avx2.tar.gz
!cp -f tmp/mmseqs/bin/mmseqs bin/mmseqs
!chmod +x bin/mmseqs
!./bin/mmseqs version


--2026-08-11 09:49:18--  https://github.com/steineggerlab/foldseek/releases/download/10-941cd33/foldseek-linux-avx2.tar.gz
Resolving github.com (github.com)... 20.205.243.166
Connecting to github.com (github.com)|20.205.243.166|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://release-assets.githubusercontent.com/github-production-release-asset/166769317/569790c8-9f90-48e1-bc4c-01d4ceac48f3?sp=r&sv=2018-11-09&sr=b&spr=https&se=2026-08-11T02%3A44%3A22Z&rscd=attachment%3B+filename%3Dfoldseek-linux-avx2.tar.gz&rsct=application%2Foctet-stream&skoid=96c2d410-5711-43a1-aedd-ab1947aa7ab0&sktid=398a6654-997b-47e9-b12b-9515b896b4de&skt=2026-08-11T01%3A43%3A45Z&ske=2026-08-11T02%3A44%3A22Z&sks=b&skv=2018-11-09&sig=r97qQXys0lcvdEzokpnfqYQ5HZD3KyDqxGYUDyP0oNo%3D&jwt=eyJ0eXAiOiJKV1QiLCJhbGciOiJIUzI1NiJ9.eyJpc3MiOiJnaXRodWIuY29tIiwiYXVkIjoicmVsZWFzZS1hc3NldHMuZ2l0aHVidXNlcmNvbnRlbnQuY29tIiwia2V5Ijoia2V5MSIsImV4cCI6MTc4NjQxNDc1OSwibmJmIjoxNzg2NDEyOTU5LCJwYXRoIjoic

## 3. 下载 SCOPe 原始数据与标签

结构包约 1GB。优先从 Hugging Face 镜像 [`caijihuize/scope40_pdbstyle`](https://huggingface.co/datasets/caijihuize/scope40_pdbstyle) 拉取。

- `pdbstyle-sel-gs-bib-40-2.08.tgz` → `tmp/pdbstyle-2.08/`
- `dir.cla.scope.2.08-stable.txt` / `dir.des.scope.2.08-stable.txt`


In [5]:
# 官方源（不稳定时改用上面的 HF）：
# !cd tmp && wget -nc https://scop.berkeley.edu/downloads/pdbstyle/pdbstyle-sel-gs-bib-40-2.08.tgz
# !cd tmp && wget -nc https://scop.berkeley.edu/downloads/parse/dir.cla.scope.2.08-stable.txt
# !cd tmp && wget -nc https://scop.berkeley.edu/downloads/parse/dir.des.scope.2.08-stable.txt

!cd tmp && wget -nc {HF_BASE}/pdbstyle-sel-gs-bib-40-2.08.tgz
!cd tmp && wget -nc {HF_BASE}/dir.cla.scope.2.08-stable.txt
!cd tmp && wget -nc {HF_BASE}/dir.des.scope.2.08-stable.txt

extract_source(skip_existing=SKIP_EXISTING)

!ls -lh tmp/pdbstyle-sel-gs-bib-40-2.08.tgz tmp/dir.cla.scope.2.08-stable.txt tmp/dir.des.scope.2.08-stable.txt
!ls -ld tmp/pdbstyle-2.08 && find tmp/pdbstyle-2.08 -name '*.ent' | wc -l


File ‘pdbstyle-sel-gs-bib-40-2.08.tgz’ already there; not retrieving.

File ‘dir.cla.scope.2.08-stable.txt’ already there; not retrieving.

File ‘dir.des.scope.2.08-stable.txt’ already there; not retrieving.

⏭️  已解压: /hpcfs/fhome/caihuize/scope40_easy/tmp/pdbstyle-2.08
-rw-r--r-- 1 caihuize hpcuser  34M Aug 11 09:50 tmp/dir.cla.scope.2.08-stable.txt
-rw-r--r-- 1 caihuize hpcuser  16M Aug 11 09:50 tmp/dir.des.scope.2.08-stable.txt
-rw-r--r-- 1 caihuize hpcuser 993M Aug 11 09:50 tmp/pdbstyle-sel-gs-bib-40-2.08.tgz
drwxr-xr-x 941 caihuize hpcuser 32768 Aug 11 09:51 tmp/pdbstyle-2.08
15177


## 4. 环境校验

确认工具与数据齐全后再分类建库。


In [6]:
checks = {
    "foldseek": FOLDSEEK_BIN,
    "mmseqs": MMSEQS_BIN,
    "source archive": SOURCE_ARCHIVE,
    "SCOP cla": SCOP_CLA,
    "SCOP des": SCOP_DES,
}
missing = []
for name, path in checks.items():
    ok = path.is_file()
    print(("✅" if ok else "❌"), f"{name}: {path}")
    if not ok:
        missing.append(name)

extracted_ok = EXTRACTED.is_dir() and any(EXTRACTED.rglob("*.ent"))
print(("✅" if extracted_ok else "❌"), f"extracted pdbstyle: {EXTRACTED}")
if not extracted_ok:
    missing.append("extracted pdbstyle")

if missing:
    raise SystemExit(f"下载/解压未完成，缺失: {missing}")
print("\n下载与解压校验通过。")


✅ foldseek: /hpcfs/fhome/caihuize/scope40_easy/bin/foldseek
✅ mmseqs: /hpcfs/fhome/caihuize/scope40_easy/bin/mmseqs
✅ source archive: /hpcfs/fhome/caihuize/scope40_easy/tmp/pdbstyle-sel-gs-bib-40-2.08.tgz
✅ SCOP cla: /hpcfs/fhome/caihuize/scope40_easy/tmp/dir.cla.scope.2.08-stable.txt
✅ SCOP des: /hpcfs/fhome/caihuize/scope40_easy/tmp/dir.des.scope.2.08-stable.txt
✅ extracted pdbstyle: /hpcfs/fhome/caihuize/scope40_easy/tmp/pdbstyle-2.08

下载与解压校验通过。


## 5. 按标签与 MODEL 数分类

对 `tmp/pdbstyle-2.08/**/*.ent`：

1. 无 ATOM/HETATM，或不在 `dir.cla` → `tmp/work/SCOPe40-other/`
2. 否则 MODEL 数 ≤1 → `SCOPe40/`（single），>1 → `SCOPe40-multi/`

写清单到 `tmp/work/structure_classification.tsv`（中间产物）。


In [7]:
manifest = classify_structures(skip_existing=SKIP_EXISTING)
print("single:", SINGLE)
print("multi:", MULTI)
print("other:", OTHER)
print("manifest:", manifest)
!head -n 5 {CLASSIFICATION}


分类进度: 1,000/15,177
分类进度: 2,000/15,177
分类进度: 3,000/15,177
分类进度: 4,000/15,177
分类进度: 5,000/15,177
分类进度: 6,000/15,177
分类进度: 7,000/15,177
分类进度: 8,000/15,177
分类进度: 9,000/15,177
分类进度: 10,000/15,177
分类进度: 11,000/15,177
分类进度: 12,000/15,177
分类进度: 13,000/15,177
分类进度: 14,000/15,177
分类进度: 15,000/15,177
✅ 分类: single=13,869, multi=1,307, other=1
single: /hpcfs/fhome/caihuize/scope40_easy/tmp/work/SCOPe40
multi: /hpcfs/fhome/caihuize/scope40_easy/tmp/work/SCOPe40-multi
other: /hpcfs/fhome/caihuize/scope40_easy/tmp/work/SCOPe40-other
manifest: /hpcfs/fhome/caihuize/scope40_easy/tmp/work/structure_classification.tsv
file	model_count	group	in_cla	scop_class
d12asa_.ent	0	single	1	d.104.1.1
d16vpa_.ent	0	single	1	d.180.1.1
d1914a1.ent	0	single	1	d.49.1.1
d1914a2.ent	0	single	1	d.49.1.1


## 6. 构建 Foldseek DB

对 `tmp/work/SCOPe40/`（single）执行 `createdb` / `lndb` / `convert2fasta`，产出 AA / 3Di FASTA。


In [8]:
db = build_foldseek_db(skip_existing=SKIP_EXISTING, threads=THREADS)
!head -n 5 {BUILD_DB}/DB_aa.fasta
!head -n 5 {BUILD_DB}/DB_di.fasta
print("foldseek db:", db)


[CMD] /hpcfs/fhome/caihuize/scope40_easy/bin/foldseek createdb /hpcfs/fhome/caihuize/scope40_easy/tmp/work/SCOPe40 /hpcfs/fhome/caihuize/scope40_easy/tmp/work/FoldseekDB/DB --threads 16
createdb /hpcfs/fhome/caihuize/scope40_easy/tmp/work/SCOPe40 /hpcfs/fhome/caihuize/scope40_easy/tmp/work/FoldseekDB/DB --threads 16 

MMseqs Version:             	941cd33ff0771cd2e3f144e3293e22a2b87e9fda
Use GPU                     	0
Path to ProstT5             	
Chain name mode             	0
Createdb extraction mode    	0
Interface distance threshold	8
Write mapping file          	0
Mask b-factor threshold     	0
Coord store mode            	2
Write lookup file           	1
Input format                	0
File Inclusion Regex        	.*
File Exclusion Regex        	^$
Threads                     	16
Verbosity                   	3

Output file: /hpcfs/fhome/caihuize/scope40_easy/tmp/work/FoldseekDB/DB
[=================================================================] 13.87K 2s 64ms
Time for merging to

## 7. 构建 MMseqs DB

用 Foldseek 导出的 `DB_aa.fasta`（ID 与 FoldseekDB / scop_lookup 一致）构建序列库。


In [9]:
mmseqs_db = build_mmseqs_db(skip_existing=SKIP_EXISTING, threads=THREADS)
print("mmseqs db:", mmseqs_db)
!ls -lh {BUILD_MMSEQS_DB} | head


[CMD] /hpcfs/fhome/caihuize/scope40_easy/bin/mmseqs createdb /hpcfs/fhome/caihuize/scope40_easy/tmp/work/FoldseekDB/DB_aa.fasta /hpcfs/fhome/caihuize/scope40_easy/tmp/work/MMseqsDB/DB --threads 16
createdb /hpcfs/fhome/caihuize/scope40_easy/tmp/work/FoldseekDB/DB_aa.fasta /hpcfs/fhome/caihuize/scope40_easy/tmp/work/MMseqsDB/DB --threads 16 

MMseqs Version:                    	8cc5ce367b5638c4306c2d7cfc652dd099a4643f
Database type                      	0
Shuffle input database             	true
Createdb mode                      	0
Write lookup file                  	1
Offset of numeric ids              	0
Threads                            	16
Compressed                         	0
Mask residues                      	0
Mask residues probability          	0.9
Mask lower case residues           	0
Mask lower letter repeating N times	0
Use GPU                            	0
Verbosity                          	3

Converting sequences
[=
Time for merging to DB_h: 0h 0m 0s 30ms
Time for mergi

## 8. 安装到 `work/`

将建好的库与 FASTA 写入：

- `work/DB/foldseek_DB`、`work/DB/mmseqs_DB`
- `work/GT_fasta/DB_aa.fasta`、`DB_di.fasta`
- `work/lable/scop_lookup.tsv`（仅此标签文件）

可选：打包 `work/scope40_work_bundle.tar.gz` 便于分发。


In [10]:
install_to_work(skip_existing=SKIP_EXISTING)
bundle = make_work_bundle(skip_existing=SKIP_EXISTING)
print("bundle:", bundle)
!ls -la {GT_FASTA_DIR}
!ls -la {DBS_DIR}
!ls -la {LABEL_DIR}


✅ scop lookup: /hpcfs/fhome/caihuize/scope40_easy/work/lable/scop_lookup.tsv  匹配=13,920 未匹配=0
✅ 已安装到 /hpcfs/fhome/caihuize/scope40_easy/work
   GT_fasta: DB_aa.fasta, DB_di.fasta
   DB: foldseek_DB, mmseqs_DB
   lable: /hpcfs/fhome/caihuize/scope40_easy/work/lable/scop_lookup.tsv
✅ bundle: /hpcfs/fhome/caihuize/scope40_easy/work/scope40_work_bundle.tar.gz (21.9 MiB)
bundle: /hpcfs/fhome/caihuize/scope40_easy/work/scope40_work_bundle.tar.gz
total 1
drwxr-xr-x  2 caihuize hpcuser    4096 Aug 11 09:53 .
drwxr-xr-x 11 caihuize hpcuser    4096 Aug 11 09:53 ..
-rw-r--r--  1 caihuize hpcuser 2791416 Aug 11 09:53 DB_aa.fasta
-rw-r--r--  1 caihuize hpcuser 2791416 Aug 11 09:53 DB_di.fasta
total 2
drwxr-xr-x  4 caihuize hpcuser 4096 Aug 11 09:53 .
drwxr-xr-x 11 caihuize hpcuser 4096 Aug 11 09:53 ..
drwxr-xr-x  2 caihuize hpcuser 4096 Aug 11 09:53 foldseek_DB
drwxr-xr-x  2 caihuize hpcuser 4096 Aug 11 09:53 mmseqs_DB
total 1
drwxr-xr-x  2 caihuize hpcuser   4096 Aug 11 09:53 .
drwxr-xr-x 11 caihu

## 9. 校验（可选）

检查 `work/DB`、`work/GT_fasta`、`work/lable/scop_lookup.tsv` 是否齐全。


In [11]:
verify_work()


✅ foldseek DB: /hpcfs/fhome/caihuize/scope40_easy/work/DB/foldseek_DB/DB
✅ mmseqs DB: /hpcfs/fhome/caihuize/scope40_easy/work/DB/mmseqs_DB/DB
✅ DB_aa.fasta: /hpcfs/fhome/caihuize/scope40_easy/work/GT_fasta/DB_aa.fasta
✅ DB_di.fasta: /hpcfs/fhome/caihuize/scope40_easy/work/GT_fasta/DB_di.fasta
✅ scop_lookup: /hpcfs/fhome/caihuize/scope40_easy/work/lable/scop_lookup.tsv
verify OK — scop_lookup 行数=13,920


## 清理临时目录

删除项目根 `tmp/` 与 `work/tmp/`（下载/解压/搜索中间文件）。产物在 `work/` 与 `bin/` 中保留。


In [12]:
cleanup_tmp(also_work_tmp=True)


🧹 已清理: /hpcfs/fhome/caihuize/scope40_easy/tmp
🧹 已清理: /hpcfs/fhome/caihuize/scope40_easy/work/tmp
